# Assignment Customer Satisfaction and Sentiment Analysis


## Objective

You are a data analyst of a consulting company that provides customer insight regarding multiple ticketing system, such as JIRA and Zoho Desk. Your team gather surveys to customers regarding their ticketing system's performance. Your role in the team is to gather reports regarding customer satisfaction and sentiment analysis into a single dashboard and present your insight.

Analyze the following metrics and other insight you can find in the dataset:

- Survey response rate
- Customer Satisfaction score (CSAT)
- Customer Effort Score (CES)
- Net Promoter Score (NPS)
- Sentiment Analysis



## Data Preparation

In [ ]:
import numpy as np
import pandas as pd
import os

pd.options.display.max_columns = 999
pd.options.display.float_format = "{:.2f}".format

### Access to Drive

Write where you put the data in google drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

# Where is your data_path
data_path = '/content/assignment_ticket_system_review.csv'

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


### Read Data

Read the file **assignment_ticket_system_review.csv**

In [ ]:
# Read Data
df = pd.read_csv(data_path)
df.head()

,id_survey,date_of_survey,ticket_system,overall_rating,customer_service,features,value_for_money,ease_of_use,likelihood_to_recommend,overall_text
0,T_02161,2024-11-20,Zendesk,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,T_00229,2024-10-06,Zendesk,3.00,4.00,3.00,3.00,2.00,6.00,Customer tickets managements
2,T_04527,2024-12-26,Zoho Desk,5.00,5.00,5.00,5.00,5.00,8.00,"After 6 months of using the Zoho desk, we shif..."
3,T_03190,2024-12-08,Zoho Desk,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,T_00644,2024-10-17,Zendesk,5.00,3.00,4.00,5.00,5.00,6.00,Pros:Zendesk has always been one of the go-to ...


The following is the dictionary for the data, survey is only valid if all of the survey questions and text review is not blank (null):

**General Information**
- id_survey: identifier for each survey
- date_of_survey: date of survey taken
- ticket_system: The name of the ticket system being reviewed (e.g. Zoho Desk)

**Survey Questions**
- overall_rating: The overall satisfaction rating given by the reviewer, ranging from 1 to 5
- customer_service: The satisfaction rating for the customer service provided by the ticket system, ranging from 1 to 5.
- features: The satisfaction rating for the features of the ticket system, ranging from 1 to 5
- value_for_money: The satisfaction rating for the value for money provided by the ticket system, ranging from 1 to 5
- ease_of_use: The rating for how easy the ticket system is to use, ranging from 1 to 5
- likelihood_to_recommend: The likelihood that the reviewer would recommend the ticket system to others, ranging from 1 to 10
- overall_text: The full text of the overall review, providing detailed feedback on the ticket system.


In [ ]:
# Check the type of data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id_survey                1462 non-null   object 
 1   date_of_survey           1462 non-null   object 
 2   ticket_system            1462 non-null   object 
 3   overall_rating           787 non-null    float64
 4   customer_service         787 non-null    float64
 5   features                 787 non-null    float64
 6   value_for_money          787 non-null    float64
 7   ease_of_use              787 non-null    float64
 8   likelihood_to_recommend  787 non-null    float64
 9   overall_text             787 non-null    object 
dtypes: float64(6), object(4)
memory usage: 114.3+ KB


### Data Cleansing

Convert the date column into proper date_time format.

In [ ]:
# Convert data type
df['date_of_survey'] = pd.to_datetime(df['date_of_survey'])

df.dtypes

,0
id_survey,object
date_of_survey,datetime64[ns]
ticket_system,object
overall_rating,float64
customer_service,float64
features,float64
value_for_money,float64
ease_of_use,float64
likelihood_to_recommend,float64
overall_text,object


Before moving on, let's take a quick look at the value range of each rating column to make sure there's nothing weird in there.

In [ ]:
rating_cols = ['overall_rating', 'customer_service', 'features',
               'value_for_money', 'ease_of_use', 'likelihood_to_recommend']

df[rating_cols].describe()

,overall_rating,customer_service,features,value_for_money,ease_of_use,likelihood_to_recommend
count,787.00,787.00,787.00,787.00,787.00,787.00
mean,4.56,3.37,4.42,4.38,4.47,7.61
std,0.64,1.14,0.72,0.82,0.73,1.72
min,1.00,-1.00,1.00,1.00,1.00,-1.00
25%,4.00,3.00,4.00,4.00,4.00,7.00
50%,5.00,3.00,5.00,5.00,5.00,8.00
75%,5.00,4.00,5.00,5.00,5.00,9.00
max,5.00,5.00,5.00,5.00,5.00,10.00


The minimum for `customer_service` and `likelihood_to_recommend` shows -1, which doesn't make sense since the rating scale starts from 1. These look like invalid entries, so I'll treat them as missing values instead of keeping them.

In [ ]:
# -1 is out of the valid rating range, replace it with NaN so it won't mess up the scoring
df[rating_cols] = df[rating_cols].replace(-1, np.nan)

df[rating_cols].describe()

,overall_rating,customer_service,features,value_for_money,ease_of_use,likelihood_to_recommend
count,787.00,785.00,787.00,787.00,787.00,786.00
mean,4.56,3.38,4.42,4.38,4.47,7.62
std,0.64,1.11,0.72,0.82,0.73,1.69
min,1.00,0.00,1.00,1.00,1.00,0.00
25%,4.00,3.00,4.00,4.00,4.00,7.00
50%,5.00,3.00,5.00,5.00,5.00,8.00
75%,5.00,4.00,5.00,5.00,5.00,9.00
max,5.00,5.00,5.00,5.00,5.00,10.00


## Survey Analysis

### Response Rate

Start by analyzing how many customers has filled the survey, indicated by whether the overall_rating is not blank.

In [ ]:
# How many customer responded to the survey?
total_survey = len(df)
total_responded = df['overall_rating'].notna().sum()

response_rate = total_responded / total_survey * 100

print(f'Total survey sent   : {total_survey}')
print(f'Total responded     : {total_responded}')
print(f'Response rate       : {response_rate:.2f}%')

Total survey sent   : 1462
Total responded     : 787
Response rate       : 53.83%


Create a new dataframe that consists only of those who have responded the survey to simplify calculating the CSAT, CES, and NPS Score.

In [ ]:
# Responded Customer
df_resp = df[df['overall_rating'].notna()].copy()
df_resp = df_resp.reset_index(drop=True)

df_resp.shape

(787, 10)

### CSAT Score

Measure the customer's overall satisfaction score (CSAT) with the following formula:

$$
CSAT = \frac{\Sigma\ total\ satisfaction\ score}{number\ of\ responded\ customer \times \max\ rating}
$$

The max rating is inserted to convert the CSAT score into percentage.

CSAT score can be classified into categories based on the result. There is no absolute threshold for each categories but the following is the common threshold:

- \>= 90%: Excellent
- 75%-90%: Good
- 60-75%: Fair
- \<60%: Poor

In [ ]:
# CSAT Score
max_rating = 5
n_resp = len(df_resp)

csat = df_resp['overall_rating'].sum() / (n_resp * max_rating) * 100

print(f'CSAT Score: {csat:.2f}%')

CSAT Score: 91.18%


Measure the satisfaction score for the following attributes:

- customer service
- features
- value for money

In [ ]:
# Satisfaction Score for Attributes
attributes = ['customer_service', 'features', 'value_for_money']

for col in attributes:
    score = df_resp[col].sum() / (n_resp * max_rating) * 100
    print(f'{col:20s}: {score:.2f}%')

customer_service    : 67.40%
features            : 88.34%
value_for_money     : 87.60%


### CES Score

Measure CES with the following formula


$$
CES = \frac{\Sigma\ total\ effort\ score}{number\ of\ responded\ customer \times \max\ rating}
$$

In [ ]:
# CES Score
# ease_of_use represents how easy (low effort) the system is to use
ces = df_resp['ease_of_use'].sum() / (n_resp * max_rating) * 100

print(f'CES Score: {ces:.2f}%')

CES Score: 89.48%


### NPS Score

To calculate the NPS score, first we must convert the **would_you_recommend** column into proper NPS Category based on the rating value:

* Promoter: Rating 9-10
* Passive: Rating 7-8
* Detractor: Rating < 7

In [ ]:
# Category NPS
def nps_category(score):
    if score >= 9:
        return 'Promoter'
    elif score >= 7:
        return 'Passive'
    else:
        return 'Detractor'

df_resp['nps_category'] = df_resp['likelihood_to_recommend'].apply(nps_category)

df_resp['nps_category'].value_counts()

,count
nps_category,
Passive,379
Promoter,251
Detractor,157



Calculate the NPS Score with the following formula

$$
NPS = \frac{Promoter - Detractor}{Total\ Survey\ Responded}
$$

In [ ]:
# NPS Score
promoter  = (df_resp['nps_category'] == 'Promoter').sum()
detractor = (df_resp['nps_category'] == 'Detractor').sum()

nps = (promoter - detractor) / n_resp * 100

print(f'Promoter : {promoter}')
print(f'Detractor: {detractor}')
print(f'NPS Score: {nps:.2f}')

Promoter : 251
Detractor: 157
NPS Score: 11.94


NPS score can be ranging from -100 (when all customers are detractor) to 100 (when all customers are promoter).

NPS Score can be classified into categories based on the following threshold:

- \>= 70: Excellent
- 50-69: Very Good
- 30-49: Good
- 0-29: Average
- \< 0: Poor

## Sentiment Analysis

Create a new dataframe with no blank overall_text.

In [ ]:
# Create new dataframe
df_text = df_resp[df_resp['overall_text'].notna()].copy()
df_text = df_text.reset_index(drop=True)

df_text.shape

(787, 11)

### Text Cleansing

In order to get more accurate sentiment, several text cleansing need to be done. However, in most of recent sentiment analysis models and algorithm, the only text cleansing needed are as follows:

* Clean double whitespace
* Clean URL/website
* Clean username (mostly in social media or digital text)

In [ ]:
import re

def cleansing_text(x):
  # clean double whitespace
  out_text = ' '.join(x.split())

  # clean url
  out_text = re.sub(r"http\S+|www\S+|https\S+", 'http', out_text)

  # clean username
  out_text = re.sub(r"@\S+", '@user', out_text)

  return(out_text)

cleansing_text(" Doesn't  dissapoint. The car       was great. It was the best car rental experiences I've had! Salute to @jone who recommend https:/rental.com")

"Doesn't dissapoint. The car was great. It was the best car rental experiences I've had! Salute to @user who recommend http"

In [ ]:
# apply cleansing to review
df_text['clean_text'] = df_text['overall_text'].apply(cleansing_text)

df_text[['overall_text', 'clean_text']].head()

,overall_text,clean_text
0,Customer tickets managements,Customer tickets managements
1,"After 6 months of using the Zoho desk, we shif...","After 6 months of using the Zoho desk, we shif..."
2,Pros:Zendesk has always been one of the go-to ...,Pros:Zendesk has always been one of the go-to ...
3,It has been very useful so far to integrate mu...,It has been very useful so far to integrate mu...
4,Pros:It's easy to use and very intuitive.We ha...,Pros:It's easy to use and very intuitive.We ha...


### Sentiment Analysis

Create a sentiment categories using algorithm of your own choice.

The reviews are in English, so I'll use VADER (Valence Aware Dictionary for sEntiment Reasoning). It's a lexicon-based model that works well for short review-style text and doesn't need any training data. It returns a compound score between -1 and 1, which I then map into Positive / Neutral / Negative using the standard thresholds (>= 0.05 positive, <= -0.05 negative, in between is neutral).

In [ ]:
!pip install vaderSentiment -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 8.6 MB/s eta 0:00:00


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = analyzer.polarity_scores(text)['compound']
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df_text['sentiment'] = df_text['clean_text'].apply(get_sentiment)

df_text[['clean_text', 'sentiment']].head()

,clean_text,sentiment
0,Customer tickets managements,Neutral
1,"After 6 months of using the Zoho desk, we shif...",Positive
2,Pros:Zendesk has always been one of the go-to ...,Positive
3,It has been very useful so far to integrate mu...,Positive
4,Pros:It's easy to use and very intuitive.We ha...,Positive


Check the number of data by sentiment.

In [ ]:
df_text['sentiment'].value_counts()

,count
sentiment,
Positive,674
Neutral,85
Negative,28


## Finalize Data for Reporting

Save the review data with NPS category and sentiment information to new csv for the dashboard.

In [ ]:
final_df = df_resp.merge(
    df_text[['id_survey', 'clean_text', 'sentiment']],
    on='id_survey',
    how='left'
)

final_df.head()

,id_survey,date_of_survey,ticket_system,overall_rating,customer_service,features,value_for_money,ease_of_use,likelihood_to_recommend,overall_text,nps_category,clean_text,sentiment
0,T_00229,2024-10-06,Zendesk,3.00,4.00,3.00,3.00,2.00,6.00,Customer tickets managements,Detractor,Customer tickets managements,Neutral
1,T_04527,2024-12-26,Zoho Desk,5.00,5.00,5.00,5.00,5.00,8.00,"After 6 months of using the Zoho desk, we shif...",Passive,"After 6 months of using the Zoho desk, we shif...",Positive
2,T_00644,2024-10-17,Zendesk,5.00,3.00,4.00,5.00,5.00,6.00,Pros:Zendesk has always been one of the go-to ...,Detractor,Pros:Zendesk has always been one of the go-to ...,Positive
3,T_04682,2024-12-28,Zoho Desk,5.00,4.00,5.00,5.00,5.00,8.00,It has been very useful so far to integrate mu...,Passive,It has been very useful so far to integrate mu...,Positive
4,T_01238,2024-11-02,Freshdesk,4.00,4.00,4.00,5.00,4.00,8.00,Pros:It's easy to use and very intuitive.We ha...,Passive,Pros:It's easy to use and very intuitive.We ha...,Positive


In [ ]:
final_df.to_csv('/content/ticket_review_final.csv', index=False)

print('Saved:', final_df.shape)

Saved: (787, 13)
